In [4]:
import pandas as pd

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

print(train.shape)

(975, 30)


In [5]:
option_cols = train.columns[2:]

rows = []

for _, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    for i in range(1, len(vals)-1):

        center = vals[i]

        if np.isnan(center):
            continue

        left = vals[i-1]
        right = vals[i+1]

        if np.isnan(left) or np.isnan(right):
            continue

        linear_pred = (left + right) / 2

        residual = center - linear_pred

        rows.append({
            "left": left,
            "right": right,
            "linear_pred": linear_pred,
            "target": residual
        })

df_res = pd.DataFrame(rows)

print(df_res.shape)

(12981, 4)


In [6]:
!pip install lightgbm -q

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

In [8]:
X = df_res[
    ["left", "right", "linear_pred"]
]

y = df_res["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train)

pred_residual = model.predict(X_valid)

pred_iv = (
    X_valid["linear_pred"].values
    + pred_residual
)

true_iv = (
    X_valid["linear_pred"].values
    + y_valid.values
)

mse = mean_squared_error(
    true_iv,
    pred_iv
)

print("Residual Model MSE:", mse)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000509 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 10384, number of used features: 3
[LightGBM] [Info] Start training from score 0.000082
Residual Model MSE: 0.00012525451559120454
